In [ ]:
import numpy as np
import xarray as xr
from datetime import datetime, timedelta
# from scipy import stats
import h5py
import json
import pickle
import glob

In [ ]:
# for plotting
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import colorcet as cc
import numpy as np
import pickle
import json
from pathlib import Path

sns.set()
sns.set_context('poster')
sns.set_style('ticks')
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = 'cmr10'
plt.rcParams["mathtext.fontset"] = 'cm'
plt.rcParams["axes.formatter.use_mathtext"] = True

In [ ]:
from sam_sat import qsatw

In [ ]:
def cloud_mask(sam3d, crit = 1.0e-12):
    tr_field = np.array(sam3d['QN'][:])
    return tr_field > crit

In [ ]:
def plume_mask(sam3d, frac = 0.5):
    tr_field = np.array(sam3d['TR01'][:])
    tr_mean = np.mean(tr_field, axis=(2,3))
    tr_stdev = np.std(tr_field, axis=(2,3))
    tr_min = .05 * np.cumsum(tr_stdev, axis=1)/(np.arange(len(tr_stdev[0,:]))+1)
    tr_limit = np.maximum(tr_mean+frac*tr_stdev, tr_min)
    return tr_mean, tr_stdev, (tr_field - tr_mean[:, :, None, None])/tr_stdev[:, :, None, None], tr_field> tr_limit[:, :, None, None]

In [ ]:
def halo_mask_1(sam3d, frac = 1.0):
    field = np.array(sam3d['QV'][:]+sam3d['QN'][:])
    fmean = np.mean(field, axis=(2,3))
    fstdev = np.std(field, axis=(2,3))
    flimit = fmean + frac*fstdev
    return field > flimit[:, :, None, None]

In [ ]:
def halo_mask_2(sam3d, frac = 1.0, diagnostics=False):
    # using qt or qv for RH?
    qv = np.array(sam3d['QV'][:])
    t_k = np.array(sam3d['TABS'][:]) 
    p_mb = np.array(sam3d['p'][:])
    p_mb = np.broadcast_to(p_mb[None, :, None, None], t_k.shape)
    qs = qsatw(t_k, p_mb)*1.0e3  # in g/kg
    field = qv / qs
    fmean = np.mean(field, axis=(2,3))
    fstdev = np.std(field, axis=(2,3))
    if diagnostics:
        for z, m, s in zip(sam3d['z'].values.flatten(), fmean.flatten(), fstdev.flatten()):
            print(z, f'{m*100:.2f}', f'{s*100:.4f}')
    flimit = fmean + frac*fstdev
    return fmean, fstdev, (field - fmean[:, :, None, None])/fstdev[:, :, None, None], field > flimit[:, :, None, None]

In [ ]:
t = 56200
d = xr.open_dataset(f'OUT_3D/BOMEX_r1_256_{t:010d}.nc')

In [ ]:
nx = 512
dx = 25.0 # m
xi = np.arange(0, nx*dx+1.0, dx)
xi = xi*1.0e-3 # km
x = (xi[:-1] + xi[1:])*0.5

In [ ]:
tr_mean, tr_stdev, ntr, pmask = plume_mask(d, frac=0.5)
# tr_mean, tr_stdev, ntr, pmask2  = plume_mask(d, frac=1.0)
hmask1 = halo_mask_1(d, frac=1.0)
rhm, rhstd, nrh, hmask2 = halo_mask_2(d, frac=1.0, diagnostics=True)
cmask = cloud_mask(d)

In [ ]:
levs = [25, 35, 50]

In [ ]:
fig = plt.figure(figsize=(6,9))
ax = fig.add_axes((0.15, 0.1, 0.8, 0.85))
ax.plot(np.mean(cmask, axis=(2,3)).flatten(), d.z[:]/1000, label='cloud', color='royalblue')
ax.plot(np.mean(pmask, axis=(2,3)).flatten(), d.z[:]/1000, label='plume', color='gray')
# ax.plot(np.mean(pmask2|cmask, axis=(2,3)).flatten(), d.z[:]/1000, label='plume (1.0)', color='gray', linestyle='dashed')
ax.plot(np.mean(hmask2, axis=(2,3)).flatten(), d.z[:]/1000, label='moist halo (RH)', color='black')
# ax.plot(np.mean(hmask1, axis=(2,3)).flatten(), d.z[:]/1000, label='halo mask (qt)', color='red', linestyle='dashed')
ax.plot(np.mean(pmask&hmask2, axis=(2,3)).flatten(), d.z[:]/1000, label='halo in plume', color='green')
# ax.plot(np.mean(pmask2&hmask2, axis=(2,3)).flatten(), d.z[:]/1000, label='halo in plume (1.0)', color='purple', linestyle='dotted')
ax.plot(np.mean((~pmask)&hmask2, axis=(2,3)).flatten(), d.z[:]/1000, label='halo outside plume', color='red')
# ax.plot(np.mean((~pmask2)&hmask2, axis=(2,3)).flatten(), d.z[:]/1000, label='halo outside plume (1.0)', color='orange', linestyle='dotted')
# for l in levs:
#     ax.axhline(y=d.z[l]*1.0e-3, color='black', linewidth=0.8, alpha=0.5)
ax.set_ylim(0, 2)
ax.set_title('Area fraction')
ax.set_ylabel('Height (km)')
plt.legend(fontsize=14)

plt.show()

In [ ]:
from matplotlib.patches import Patch

for iz in levs:

    fig, axs = plt.subplots(1, 1, figsize=(12, 12))
    axs.set_aspect('equal')
    levels = [ 0.5, 1.5]


    colors = ['none', ('red', 0.7), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, ((~pmask)&hmask2)[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('dimgray', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm2 = axs.pcolormesh(xi, xi, (pmask&(~cmask)&(~hmask2))[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    # colors = ['none', ('blue', 0.5), 'none']
    # cmap = mpl.colors.ListedColormap(colors)
    # norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    # cm3 = axs.pcolormesh(xi, xi, (pmask&(~pmask2)&hmask2&(~cmask))[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('limegreen', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm5 = axs.pcolormesh(xi, xi, ((~cmask)&pmask&hmask2)[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('royalblue', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm1 = axs.pcolormesh(xi, xi, cmask[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    axs.set_title(f'z={d.z[iz]:.1f} m')
    axs.set_xlabel('x (km)')
    axs.set_ylabel('y (km)')

    legend_elements = [
        Patch(facecolor='royalblue', alpha=1.0, label='Cloud'),
        Patch(facecolor='limegreen', alpha=1.0, label='Moist halo in plume (no cloud)'),
        Patch(facecolor='dimgray', alpha=1.0, label='Plume (no halo, no cloud)'),
        Patch(facecolor='red', alpha=1.0, label='Moist halo outside plume'),
    ]
    axs.legend(handles=legend_elements, loc='upper right', fontsize=16)
    
    plt.show()

In [ ]:
from matplotlib.patches import Patch

for iz in levs:

    fig, axs = plt.subplots(1, 1, figsize=(12, 12))
    axs.set_aspect('equal')


    clevels = np.arange(-1.0, 1.2, 0.2)
    # print(nrh[0, iz, :, :].min(), nrh[0, iz, :, :].max())
    nrh_plot = axs.contourf(x, x, np.where((~hmask2[0, iz, :, :]) & (~pmask[0, iz, :, :]), nrh[0, iz, :, :], np.nan), cmap=mpl.cm.bwr, levels=clevels, extend='both')
    plt.colorbar(nrh_plot, ax=axs, label='Normalized RH')

    colors = ['none', ('black', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, (pmask)[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('green', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, cmask[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    # ntr_plot = axs.contourf(x, x, ntr[0, iz, :, :], colors='white', levels=[0.5])

    axs.set_title(f'z={d.z[iz]:.1f} m')
    axs.set_xlabel('x (km)')
    axs.set_ylabel('y (km)')

    plt.show()

In [ ]:
from matplotlib.patches import Patch

for iz in levs:

    fig, axs = plt.subplots(1, 1, figsize=(12, 12))
    axs.set_aspect('equal')

    clevels = np.arange(-1.0, 1.1, 0.25)
    # print(nrh[0, iz, :, :].min(), nrh[0, iz, :, :].max())
    # cmap = deepcopy(mpl.cm.bwr)
    # cmap.set_over('green')
    nrh_plot = axs.contourf(x, x, np.where((~pmask[0, iz, :, :]), nrh[0, iz, :, :], np.nan), cmap=mpl.cm.bwr, levels=clevels, extend='both')
    plt.colorbar(nrh_plot, ax=axs, label='Normalized RH')

    colors = ['none', ('palegoldenrod', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, pmask[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('indigo', 1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, cmask[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    colors = ['none', ('limegreen',1.0), 'none']
    cmap = mpl.colors.ListedColormap(colors)
    norm = mpl.colors.BoundaryNorm(levels, ncolors=cmap.N, extend="both")
    cm4 = axs.pcolormesh(xi, xi, ((~pmask)&hmask2)[0, iz, :, :], norm=norm, cmap=cmap, shading='flat')

    # ntr_plot = axs.contour(x, x, ntr[0, iz, :, :], colors='indigo', levels=[0.1], linewidths=1)
    # ntr_plot = axs.contour(x, x, ntr[0, iz, :, :], colors='purple', levels=[0.0], linewidths=1)

    axs.set_title(f'z={d.z[iz]:.1f} m')
    axs.set_xlabel('x (km)')
    axs.set_ylabel('y (km)')

    plt.show()